# 04c — ETA còn lại theo LaDe · Protocol P

Đây là bài toán ETA chính. Mốc dự đoán là lúc shipper vừa giao xong một đơn.
Mỗi mẫu là một cặp gồm mốc dự đoán và một đơn còn cầm.

### 1. Xem đúng bảy cột nguồn từ Bronze

Chạy trong Jupyter Docker sau notebook 01. `ds` và `courier_id` dùng để nhóm
quỹ đạo. Các mốc nhận và giao quyết định ứng viên và nhãn.

In [ ]:
import sys

sys.path.insert(0, "/home/jovyan/pipeline")
import steps

spark = steps.get_spark("Delivery - Protocol P")

In [ ]:
from src.eta import paper_protocol

spark.read.format("delta").load(steps.BRONZE).select(*paper_protocol.INPUT_COLUMNS).show(5, truncate=False)

### 2. Làm sạch, dựng cặp và chia tập

1. Bỏ trùng theo bảy cột, đổi giờ sang phút trong ngày. Khi ngày nhận và giao
   khác nhau, quy trình tái lập trừ 1.440 vào phút nhận.
2. Bỏ điểm ở giữa hai đoạn vượt 500 m/phút, rồi bỏ điểm lặp vị trí trong dưới một phút.
3. Dùng cửa sổ 24 đơn, bước trượt 12. Giữ cặp có nhãn còn lại lớn hơn 0 đến 120 phút
   và cần ít nhất hai ứng viên. `paper_protocol.build_samples()` áp dụng quy tắc
   cửa sổ tương ứng với từng tập.
4. Chia theo ngày `ds` và bỏ hai ngày ranh giới theo thiết kế tái lập.

Kết quả cell cho biết số dòng còn lại sau làm sạch và số cặp ở mỗi tập.
MAE của P được đọc trong cohort này. `spd`, `ntodo` và `rank` có phụ thuộc
dữ liệu tương lai. [Kiểm kê leakage](../docs/leak_inventory.md)

In [1]:
result = steps.build_eta_paper(spark, seeds=(42, 43, 44))
result

{'run_id': '6a15165022fa4cfbbf6eb35362aac5a1',
 'raw': 206431,
 'drop_step_A': 368,
 'drop_step_C': 0,
 'kept': 206063,
 'pairs_candidate': 1915541,
 'pairs_removed_by_label_filter': 0.4816,
 'removed_label_zero': 0.0029,
 'removed_label_over_120': 0.4787,
 'pairs': {'train': 567672, 'validation': 202889, 'test': 198463}}

### 3. Xem điểm của median, SPEED và LightGBM

SPEED lấy khoảng cách chia tốc độ. LightGBM fit trên train, dùng validation
để dừng sớm và báo điểm trên test. Bảng ghi mô hình, seed, thước đo và giá trị.

In [2]:
metrics = spark.read.format("delta").load(steps.ETA_PAPER_METRICS)
metrics.orderBy("model", "seed", "metric").show(100, truncate=False)

+--------------------------------+------------+----+-----------+------------------+
|run_id                          |model       |seed|metric     |value             |
+--------------------------------+------------+----+-----------+------------------+
|6a15165022fa4cfbbf6eb35362aac5a1|lightgbm    |42  |acc20      |59.147548913399476|
|6a15165022fa4cfbbf6eb35362aac5a1|lightgbm    |42  |acc30      |76.18397383895234 |
|6a15165022fa4cfbbf6eb35362aac5a1|lightgbm    |42  |mae        |20.68829787620978 |
|6a15165022fa4cfbbf6eb35362aac5a1|lightgbm    |42  |rmse_paper |24.95309314098919 |
|6a15165022fa4cfbbf6eb35362aac5a1|lightgbm    |42  |rmse_pooled|27.13181459602439 |
|6a15165022fa4cfbbf6eb35362aac5a1|lightgbm    |43  |acc20      |59.22716073021168 |
|6a15165022fa4cfbbf6eb35362aac5a1|lightgbm    |43  |acc30      |76.17893511636929 |
|6a15165022fa4cfbbf6eb35362aac5a1|lightgbm    |43  |mae        |20.671433322577247|
|6a15165022fa4cfbbf6eb35362aac5a1|lightgbm    |43  |rmse_paper |24.933773083